<a href="https://colab.research.google.com/github/joolsa/PersonaBench/blob/main/Notebooks/PD05a_LLM_illustration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM-Based Persona Generation

This notebook demonstrates how to use Large Language Models to generate detailed personas systematically.

---

## STEP 1: Skeletal Persona Generation

First, let's set up the environment and establish our LLM connection:

In [ ]:
import pandas as pd
import numpy as np
# import guidance
import uuid
import openai
import json

OPENAI_API_KEY='sk-<your key here>'
# guidance.llm = guidance.llms.OpenAI(
#     "gpt-4", token=OPENAI_API_KEY)
openai.api_key = OPENAI_API_KEY

from tenacity import (
    retry,
    stop_after_attempt,
    wait_random_exponential,
)  # for exponential backoff

### Setup API Functions

In [ ]:
models = openai.Model.list()

@retry(wait=wait_random_exponential(min=1, max=5), stop=stop_after_attempt(44))
def chat_completion_with_backoff(**kwargs):
    print("trying")
    return openai.ChatCompletion.create(**kwargs)


# Example
# message_input = [
#     {
#         "role":"system",
#         "content": "You are a helpful assistant to a social sciences researcher"},
#     {
#         "role": "user",
#         "content": f'{type_input} \n Provide the output in a json array, with each dict containing only the following keys: "index", "name", "age",  "occupation",  "background",  "details"; \n Index starts at 1. Start your response with the open square bracket ['
#         }]

### Define Persona Categories

In [ ]:
number = 30
type_list = [
    f'Generate {number} male personas that are addicted to gambling.',
    f'Generate {number} female personas that are addicted to gambling.',
    f'Generate {number} personas that are addicted to gambling.',
    f'Generate {number} male personas that are addicted to online shopping.',
    f'Generate {number} female personas that are addicted to online shopping.',
    f'Generate {number} personas that are addicted to online shopping.',
    f'Generate {number} male personas that are addicted to alcohol.',
    f'Generate {number} female personas that are addicted to alcohol.',
    f'Generate {number} personas that are addicted to alcohol.',
    f'Generate {number} male personas that are addicted to opioids.',
    f'Generate {number} female personas that are addicted to opioids.',
    f'Generate {number} personas that are addicted to opioids.',
    f'Generate {number} male personas that are addicted to social media.',
    f'Generate {number} female personas that are addicted to social media.',
    f'Generate {number} personas that are addicted to social media.'
]

### Test API Connection

In [ ]:
# create a chat completion
chat_completion = openai.ChatCompletion.create(model="gpt-3.5-turbo", messages=[{"role": "user", "content": "Hello world"}])

# print the chat completion
print(chat_completion.choices[0].message.content)

### Generate Skeletal Personas

In [ ]:
print("start 1")
existing_results_file = 'brief_personas_only.json'
# read existing results file
try:
    with open(existing_results_file, 'r') as file:
        results = json.load(file)
except FileNotFoundError:
    results = {'results': []}

output_list = []
for type_input in type_list:
    print (type_input)
    message = f'{type_input} \n Provide the output in a json array, with each dict containing only the following keys: "index", "name", "age",  "occupation",  "background",  "details"; \n Index starts at 1. Start your response with the open square bracket ['
    print(message)

    chat_completion = chat_completion_with_backoff(model="gpt-4",
                                                   messages=[
                                                    {"role":"system", "content": "You are a helpful assistant to a social sciences researcher"},
        {
        "role": "user",
        "content": message
        }], temperature=1)

    try:
        personas = json.loads(chat_completion['choices'][0]['message']['content'])
    except:
        personas =chat_completion['choices'][0]['message']['content']

    print(personas)

    output_list.append(personas)
    results['results'].append({
        'type_input': type_input,
        'personas': personas
    })

     # save results to file
    with open(existing_results_file, 'w') as file:
        json.dump(results, file, indent=4)

---

## STEP 2: Expanded Persona Generation

Now we'll take the skeletal personas and expand them into detailed profiles.

### Setup for Expansion

In [ ]:
import pandas as pd
import numpy as np
# import guidance
import uuid
import openai
import json

OPENAI_API_KEY='sk-<your key here>'
# guidance.llm = guidance.llms.OpenAI(
#     "gpt-4", token=OPENAI_API_KEY)
openai.api_key = OPENAI_API_KEY

from tenacity import (
    retry,
    stop_after_attempt,
    wait_random_exponential,
)  # for exponential backoff

### Setup Expansion Functions

In [ ]:
models = openai.Model.list()

@retry(wait=wait_random_exponential(min=1, max=60), stop=stop_after_attempt(6))
def chat_completion_with_backoff(**kwargs):
    print("trying")
    return openai.ChatCompletion.create(**kwargs)

# Example
# message_input = [
#     {
#         "role":"system",
#         "content": "You are a helpful assistant to a social sciences researcher"
#     },
#     {   "role": "user",
#         "content": "Expand on the following summary persona. Ensure that all the information provided is used in your expanded persona. \n" + persona_short
#     }
# ]

### Test Expansion

In [ ]:
# Test 1

chat_completion = chat_completion_with_backoff(model="gpt-4",
                                                   messages=[
                                                    {"role":"system", "content": "You are a helpful assistant to a social sciences researcher"},
        {
        "role": "user",
        "content": "Expand on the following summary persona. Ensure that all the information provided is used i your expanded persona. \nPersona of someone addicted to online shopping.; name; Diana Dawson; age; 36; occupation; Journalist; background; Single and lives with her dog; details; Online shopping started with work-related shopping, expanded into personal spending"
        }], temperature=1, max_tokens=2000)

chat_completion['choices'][0]['message']['content']

### Process All Skeletal Personas for Expansion

In [ ]:
existing_results_file = 'result_personas.json'

# read existing results file
with open(existing_results_file, 'r') as file:
    results = json.load(file)

for result in results['results']:
    print (result['type_input'])
    substring = result['type_input'].split("that are ")[1]

    for persona in result['personas']:
        if 'done' not in persona:
            # print keys and values, separated by semi-colons
            persona_list = []
            persona_list.append('Persona of someone ' + substring)
            print(persona['index'])
            for key, value in persona.items():
                if key!='index':
                    persona_list.append(f"{key}; {value}")

            persona_short = '; '.join(persona_list)

            print(persona_short)

            message = "Expand on the following summary persona. Ensure that all the information provided is used in your expanded persona. \n" + persona_short

            chat_completion = chat_completion_with_backoff(model="gpt-4",
                messages=[
                    {"role":"system", "content": "You are a helpful assistant to a social sciences researcher"},
                    {"role": "user",
                    "content": message
        }], temperature=1, max_tokens=2000)
            expansion = chat_completion['choices'][0]['message']['content']

            # add 'done' and 'expansion' keys
            persona['done'] = True
            persona['expansion'] = expansion
            print(expansion)

            # save results to file
            with open(existing_results_file, 'w') as file:
                json.dump(results, file, indent=4)

---

## Summary

This notebook demonstrates a two-step process for LLM-based persona generation:

1. **Skeletal Generation**: Create basic persona outlines with key attributes
2. **Expansion**: Use LLM to develop detailed, comprehensive persona descriptions

The approach uses structured prompts and JSON formatting to ensure consistent output that can be processed programmatically.